In [1]:
!pip install selenium requests pandas Pillow beautifulsoup4 lxml

 # Pillow
  Used to work with images

Open
Resize
Save

In [ ]:
import time, re, requests, pandas as pd, os
from PIL import Image # for image  to open, process, and save images or we can resize the imge 
from io import BytesIO # lets you  file like opject make the  scrping  more fater  than  usual 
from bs4 import BeautifulSoup
from selenium import webdriver # for control  my website   like firefox 
from selenium.webdriver.chrome.service import Service # for  manages my web   and make sure working good 
from selenium.webdriver.chrome.options import Options # customize browser behavior   like no gui when  scraping 
from selenium.webdriver.common.by import By # to  locate  element's on  my webpage  like class , id , tag  for img
from selenium.webdriver.support.ui import WebDriverWait # wait until something happens
from selenium.webdriver.support import expected_conditions as EC # what you are waiting for   like first image some thing like this 
import shutil


In [ ]:
options = Options()
options.binary_location = "/usr/bin/chromium" # location of chromium on my pc 
#options.add_argument("--headless=new")
options.add_argument("--no-sandbox")# bec i 'm linux 
options.add_argument("--disable-dev-shm-usage") # bec i 'm linux 
options.add_argument("--window-size=1280,900") # set a browser reslution
"""
Websites can detect that you are using a bot (Selenium) and block you.
 These lines try to hide the fact that you are a bot:
""" 
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_argument("user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
chromedriver_path = shutil.which("chromedriver") or "/usr/lib/chromium/chromedriver"
driver = webdriver.Chrome(service=Service(chromedriver_path), options=options)
print("Driver ready ✅")

Driver ready ✅


In [ ]:
driver.get("https://www.houzz.com/photos/living-room-ideas-phbr0-bp~t_718")
WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, "img[src]"))) # here  we wait to see EC    image tage   u got i  think  
time.sleep(4)
print("Page loaded ✅  Title:", driver.title)

Page loaded ✅  Title: 75 Living Room Ideas You'll Love - April, 2026 | Houzz


In [ ]:
def collect_page(driver, room_type):
    for scroll in range(0, 6000, 600):
        driver.execute_script(f"window.scrollTo(0, {scroll});") # that's as  js code  to scroll in page 
        time.sleep(0.3)
    time.sleep(2) # save  for me   to make sure page is  fully loaded 


    ## for filters  

    soup    = BeautifulSoup(driver.page_source, "lxml")
    results = [] # store the final data 
    seen    = set() # for no  dublicate 
    # like here it's bad word if u see it ignore i  dont nneed it 
    SKIP = ("save","view","add to","sign in","join","explore","browse",
            "see more","load more","similar","get","find","discover",
            "shop","click","learn","read")

    #  good word if u see one of them  need it  
    ROOM_KEYS = {"floor","wall","ceiling","kitchen","bedroom","bathroom",
                 "living","dining","office","cabinet","countertop","island",
                 "design","photo","example","style","space","room","layout",
                 "storage","renovation","remodel","transitional","modern",
                 "traditional","contemporary","farmhouse","light","window"}
    #  Finding Images 
    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src") or ""
        if not src.startswith("http"): continue #
        """
        loop for every singel tage colled  <img>  then check the url  
        here i searc h for url  if u got it from src  ->  stander  attribure  it's ok of  
        not  check this  data-src  -> that's for lazy-loding 
        if u got url and not contenue  with  http  skip it  not url 
        """

        m = re.search(r"-w(\d+)-", src)
        if not m or int(m.group(1)) < 300: continue
        if any(s in src for s in ["logo","icon","avatar","profile"]): continue
        if src in seen: continue
        """
         here we use   r   - >  regex  men regular expression  specific pattern inside the image URL.
        Imagine src is:
       "https://site.com/image-w800-h600.jpg" 

        Search: The regex scans the string. 
        Match: It finds -w800-. 
        Capture: Because of the parentheses (\d+), it extracts 800 into a special group. 
        Result: m becomes a "Match object". 

            Scenario A (Width 800):

                Is 800 < 300? No.
                Result: Keep the image.
                

            Scenario B (Width 50):

                Is 50 < 300? Yes.
                Result: Skip the image (it's too small, likely a thumbnail or icon).
                

         اhere we skip  any think  like this    logo , icon , avataer   bec ehn  i scrap the data
         i see some image    for  profile    and i don't need it 
        if any(s in src for s in ["logo","icon","avatar","profile"]): continue
        if src in seen: continue


        """
        # Part 4: Finding the Caption (The "Bubble Up") 
        caption = None
        node = img

        for _ in range(6):
            node = node.parent
            if node is None: break

            for child in node.find_all(["p","span","div","a"], recursive=False):
                text = child.get_text(separator=" ", strip=True)
                words = text.split()

                if len(words) < 10: continue
                if text.lower().startswith(SKIP): continue #  -> pad word skip 

                lower_count = sum(1 for w in words if w.islower())
                if lower_count < 5: continue

                if not set(text.lower().split()) & ROOM_KEYS: continue

                caption = text
                break

            if caption: break

        if not caption: continue
        #   Saving the Data 
        title = (img.get("alt") or "").strip() or " ".join(caption.split()[:6])
        seen.add(src)
        results.append({"title":title,"description":caption,
                        "image_url":src,"room_type":room_type})
    return results

# test + cehck all working good or not  


In [6]:
page_data = collect_page(driver, "living-room")
print(f"Page 1: {len(page_data)} good captions")
for item in page_data[:5]:
    nw = len(item["description"].split())
    print(f"  [{nw} words] {item['description'][:85]}")

Page 1: 40 good captions
  [22 words] Inspiration for a modern gray floor living room remodel in Orange County with beige w
  [19 words] Example of a transitional light wood floor and beige floor living room design in New 
  [30 words] Living room - traditional open concept dark wood floor and brown floor living room id
  [1024 words] Living Room Ideas A living room can serve many different functions, from a formal sit
  [1024 words] Living Room Ideas A living room can serve many different functions, from a formal sit


In [ ]:
URLS  = ['https://www.houzz.com/photos/living-room-ideas-phbr0-bp~t_718']
PAGES = 400
TARGET = 4000
all_data = []
all_seen = set() # faster than  list 
print("Room:", "living-room", "| Target:", TARGET)
"""
for  style  we have alot of style     

"""
for ui, base_url in enumerate(URLS):
    style = base_url.split("~s_")[-1] if "~s_" in base_url else "main"
    print(f"\n  URL {ui+1}/{len(URLS)} — {style}")
    empty = 0
    for page in range(1, PAGES+1):
        if len(all_data) >= TARGET: # for erly stop  
            print("  ✅ Target reached"); break
        """
        here  we open  a  page and use  WebDriverWait  to make sure the page are  loaded     
        bec it's javascript 
        
        
        """
        
        driver.get(f"{base_url}?pg={page}")
        print(f"    Page {page:03d}", end=" → ")
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "img[src]")))
        except:
            print("TIMEOUT"); empty += 1
            if empty >= 3: print("  next URL"); break
            continue











        # get the new data  for our main loob     to save it 
        new = [i for i in collect_page(driver, "living-room") if i["image_url"] not in all_seen]
        for i in new: all_seen.add(i["image_url"])
        all_data.extend(new)
        print(f"{len(new):3d} new | total: {len(all_data)}")
        empty = 0 if new else empty + 1
        if empty >= 5: print("  No new — next URL"); break
        time.sleep(1.5)
    if len(all_data) >= TARGET: break

print(f"\n✅ Done. Total: {len(all_data)} images")

Room: living-room | Target: 4000

  URL 1/1 — main
    Page 001 →  40 new | total: 40
    Page 002 →  40 new | total: 80
    Page 003 →  20 new | total: 100
    Page 004 →  20 new | total: 120
    Page 005 →  20 new | total: 140
    Page 006 →  20 new | total: 160
    Page 007 →  20 new | total: 180
    Page 008 →   0 new | total: 180
    Page 009 →  40 new | total: 220
    Page 010 →  20 new | total: 240
    Page 011 →  20 new | total: 260
    Page 012 →  20 new | total: 280
    Page 013 →  20 new | total: 300
    Page 014 →  20 new | total: 320
    Page 015 →  20 new | total: 340
    Page 016 →  20 new | total: 360
    Page 017 →  20 new | total: 380
    Page 018 →  20 new | total: 400
    Page 019 →   0 new | total: 400
    Page 020 →  40 new | total: 440
    Page 021 →   0 new | total: 440
    Page 022 →  20 new | total: 460
    Page 023 →  40 new | total: 500
    Page 024 →  20 new | total: 520
    Page 025 →  20 new | total: 540
    Page 026 →  20 new | total: 560
    Page 027 → 

In [8]:
driver.quit()
print('Driver closed ✅')

Driver closed ✅


In [ ]:
df = pd.DataFrame(all_data).drop_duplicates(subset=["image_url"]).reset_index(drop=True) # this i er delete  row  number    5     make spase  for it  naa  we 'll resrte the  index for it 
df["wc"] = df["description"].str.split().str.len() # to count all  word  in 1 row 
print(f"Total: {len(df)} | Min words: {df['wc'].min()} | Avg: {df['wc'].mean():.1f} | All≥10: {(df['wc']>=10).all()} ✅")
df.head()

Total: 4007 | Min words: 10 | Avg: 129.1 | All≥10: True ✅


,title,description,image_url,room_type,wc
0,Beach House,Inspiration for a modern gray floor living roo...,https://st.hzcdn.com/fimgs/a9c154550615785a_96...,living-room,22
1,Hoboken Apartment,Example of a transitional light wood floor and...,https://st.hzcdn.com/fimgs/c7e1300c05d6aba1_74...,living-room,19
2,English Tudor,Living room - traditional open concept dark wo...,https://st.hzcdn.com/fimgs/48e140de060b252c_64...,living-room,30
3,WAYCOOL Contemporary Living,Living Room Ideas A living room can serve many...,https://st.hzcdn.com/fimgs/47d1899b04fdffbc_55...,living-room,1022
4,Greentree Project,Living Room Ideas A living room can serve many...,https://st.hzcdn.com/fimgs/6de19ae805735cfe_64...,living-room,1022


In [ ]:
os.makedirs("images/living-room", exist_ok=True)
H = {"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36"}#  it's  fake prowser  to not  blocks  us 
lp, fail = [], 0  # if  some file not save that's  c checker 
for i, row in df.iterrows():
    fname = "images/living-room/living-room_" + str(i).zfill(5) + ".jpg"  #  for saving image    like  0012   but in  5 00001   
    try:
        r = requests.get(row["image_url"], headers=H, timeout=15)
        if r.status_code == 200:
            Image.open(BytesIO(r.content)).convert("RGB").save(fname)  # here  we just  nned  rgp not rbgd  or rbga   
            lp.append(fname)
        else: lp.append(None); fail+=1
    except: lp.append(None); fail+=1
    time.sleep(0.4)
df["local_path"] = lp
print(f"Downloaded: {len(df)-fail} | Failed: {fail}")

Downloaded: 4007 | Failed: 0


In [11]:
df = df.dropna(subset=["local_path"]).drop(columns=["wc"],errors="ignore").reset_index(drop=True)
print(f"Final: {len(df)} images")
df.head()

Final: 4007 images


,title,description,image_url,room_type,local_path
0,Beach House,Inspiration for a modern gray floor living roo...,https://st.hzcdn.com/fimgs/a9c154550615785a_96...,living-room,images/living-room/living-room_00000.jpg
1,Hoboken Apartment,Example of a transitional light wood floor and...,https://st.hzcdn.com/fimgs/c7e1300c05d6aba1_74...,living-room,images/living-room/living-room_00001.jpg
2,English Tudor,Living room - traditional open concept dark wo...,https://st.hzcdn.com/fimgs/48e140de060b252c_64...,living-room,images/living-room/living-room_00002.jpg
3,WAYCOOL Contemporary Living,Living Room Ideas A living room can serve many...,https://st.hzcdn.com/fimgs/47d1899b04fdffbc_55...,living-room,images/living-room/living-room_00003.jpg
4,Greentree Project,Living Room Ideas A living room can serve many...,https://st.hzcdn.com/fimgs/6de19ae805735cfe_64...,living-room,images/living-room/living-room_00004.jpg


In [12]:
df.to_csv("living_room_dataset.csv", index=False)
print("Saved → living_room_dataset.csv ✅")

Saved → living_room_dataset.csv ✅


In [13]:
pd.read_csv("living_room_dataset.csv")

,title,description,image_url,room_type,local_path
0,Beach House,Inspiration for a modern gray floor living roo...,https://st.hzcdn.com/fimgs/a9c154550615785a_96...,living-room,images/living-room/living-room_00000.jpg
1,Hoboken Apartment,Example of a transitional light wood floor and...,https://st.hzcdn.com/fimgs/c7e1300c05d6aba1_74...,living-room,images/living-room/living-room_00001.jpg
2,English Tudor,Living room - traditional open concept dark wo...,https://st.hzcdn.com/fimgs/48e140de060b252c_64...,living-room,images/living-room/living-room_00002.jpg
3,WAYCOOL Contemporary Living,Living Room Ideas A living room can serve many...,https://st.hzcdn.com/fimgs/47d1899b04fdffbc_55...,living-room,images/living-room/living-room_00003.jpg
4,Greentree Project,Living Room Ideas A living room can serve many...,https://st.hzcdn.com/fimgs/6de19ae805735cfe_64...,living-room,images/living-room/living-room_00004.jpg
...,...,...,...,...,...
4002,Custom Atlanta Home,Large transitional formal and enclosed light w...,https://st.hzcdn.com/fimgs/5d01892209774ab2_37...,living-room,images/living-room/living-room_04002.jpg
4003,Clearbrook,European inspired transitional design home in ...,https://st.hzcdn.com/fimgs/b2a10f770ab56e7a_23...,living-room,images/living-room/living-room_04003.jpg
4004,"Big open living room with Monterey, Cabana eng...","Big open living room with Monterey, Cabana eng...",https://st.hzcdn.com/fimgs/5511d16905919000_94...,living-room,images/living-room/living-room_04004.jpg
4005,TimberHouse,"The 7,600 square-foot residence was designed f...",https://st.hzcdn.com/fimgs/a541752c03e261d7_37...,living-room,images/living-room/living-room_04005.jpg
